# Customer Retention & Churn Analysis

Future Interns - Data Science & Analytics Internship


In [1]:
import pandas as pd
import numpy as np

# Load Dataset

In [2]:
df = pd.read_csv("../data/telco_churn.csv")

df.head()

,Unnamed: 0,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,7590-VHVEG,Female,False,True,False,1,False,NaN,DSL,...,False,False,False,False,Month-to-month,True,Electronic check,29.850000,29.850000381469727,False
1,1,5575-GNVDE,Male,False,False,False,34,True,False,DSL,...,True,False,False,False,One year,False,Mailed check,56.950001,1889.5,False
2,2,3668-QPYBK,Male,False,False,False,2,True,False,DSL,...,False,False,False,False,Month-to-month,True,Mailed check,53.849998,108.1500015258789,True
3,3,7795-CFOCW,Male,False,False,False,45,False,NaN,DSL,...,True,True,False,False,One year,False,Bank transfer (automatic),42.299999,1840.75,False
4,4,9237-HQITU,Female,False,False,False,2,True,False,Fiber optic,...,False,False,False,False,Month-to-month,True,Electronic check,70.699997,151.64999389648438,True


# Dataset Information

In [3]:
print("Dataset Shape:", df.shape)

df.info()

Dataset Shape: (5043, 22)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5043 entries, 0 to 5042
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        5043 non-null   int64  
 1   customerID        5043 non-null   object 
 2   gender            5043 non-null   object 
 3   SeniorCitizen     5043 non-null   object 
 4   Partner           5043 non-null   object 
 5   Dependents        5043 non-null   object 
 6   tenure            5043 non-null   int64  
 7   PhoneService      5043 non-null   object 
 8   MultipleLines     4774 non-null   object 
 9   InternetService   5043 non-null   object 
 10  OnlineSecurity    4392 non-null   object 
 11  OnlineBackup      4392 non-null   object 
 12  DeviceProtection  4392 non-null   object 
 13  TechSupport       4392 non-null   object 
 14  StreamingTV       4392 non-null   object 
 15  StreamingMovies   4392 non-null   object 
 16  Contract        

# Missing Value Analysis

In [4]:
df.isnull().sum()

Unnamed: 0            0
customerID            0
gender                0
SeniorCitizen         0
Partner               0
Dependents            0
tenure                0
PhoneService          0
MultipleLines       269
InternetService       0
OnlineSecurity      651
OnlineBackup        651
DeviceProtection    651
TechSupport         651
StreamingTV         651
StreamingMovies     651
Contract              0
PaperlessBilling      0
PaymentMethod         0
MonthlyCharges        0
TotalCharges          5
Churn                 1
dtype: int64

# Churn Distribution

In [5]:
df["Churn"].value_counts()

False    2219
No       1487
True      780
Yes       556
Name: Churn, dtype: int64

# Data Cleaning

In [6]:
# Remove unnecessary column

df = df.drop("Unnamed: 0", axis=1)

In [13]:
df["Churn"] = df["Churn"].astype(str)

df["Churn"] = df["Churn"].replace({
    "True": "Yes",
    "False": "No"
})

print(df["Churn"].value_counts())

No     3706
Yes    1336
Name: Churn, dtype: int64


In [8]:
# Remove rows with missing Churn

df = df.dropna(subset=["Churn"])

In [9]:
# Fill missing categorical values

categorical_cols = [
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

for col in categorical_cols:
    df[col].fillna("No Service", inplace=True)

In [10]:
# Fill TotalCharges missing values

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

df["TotalCharges"].fillna(
    df["TotalCharges"].median(),
    inplace=True
)

In [11]:
print(df.isnull().sum())

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


# Customer Churn KPI Analysis

In [14]:
total_customers = len(df)

churned_customers = len(df[df["Churn"] == "Yes"])

retained_customers = len(df[df["Churn"] == "No"])

churn_rate = (churned_customers / total_customers) * 100

retention_rate = (retained_customers / total_customers) * 100

print("Total Customers:", total_customers)
print("Churned Customers:", churned_customers)
print("Retained Customers:", retained_customers)
print("Churn Rate:", round(churn_rate, 2), "%")
print("Retention Rate:", round(retention_rate, 2), "%")

Total Customers: 5042
Churned Customers: 1336
Retained Customers: 3706
Churn Rate: 26.5 %
Retention Rate: 73.5 %


# Contract Type vs Churn Analysis

In [15]:
contract_churn = pd.crosstab(
    df["Contract"],
    df["Churn"]
)

print(contract_churn)

Churn             No   Yes
Contract                  
Month-to-month  1560  1184
One year         933   122
Two year        1213    30


In [16]:
contract_churn_pct = pd.crosstab(
    df["Contract"],
    df["Churn"],
    normalize="index"
) * 100

round(contract_churn_pct,2)

Churn,No,Yes
Contract,,
Month-to-month,56.85,43.15
One year,88.44,11.56
Two year,97.59,2.41


# Payment Method vs Churn Analysis

In [18]:
payment_churn = pd.crosstab(
    df["PaymentMethod"],
    df["Churn"]
)

payment_churn

Churn,No,Yes
PaymentMethod,,
Bank transfer (automatic),927,198
Credit card (automatic),922,168
Electronic check,941,758
Mailed check,916,212


# Gender vs Churn Analysis

In [19]:
gender_churn = pd.crosstab(
    df["gender"],
    df["Churn"]
)

gender_churn

Churn,No,Yes
gender,,
Female,1823,661
Male,1883,675


# Customer Lifetime (Tenure) Analysis

In [20]:
df.groupby("Churn")["tenure"].mean()

Churn
No     37.733675
Yes    18.241766
Name: tenure, dtype: float64

In [21]:
df.groupby("Churn")["MonthlyCharges"].mean()

Churn
No     61.429682
Yes    75.211003
Name: MonthlyCharges, dtype: float64

In [22]:
df.to_csv("../data/telco_churn_cleaned.csv", index=False)

print("Dataset Saved Successfully")

Dataset Saved Successfully
